In [ ]:
import os
import sys
from datetime import datetime
from zoneinfo import ZoneInfo

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

In [ ]:
import sys
sys.path.append("../src")

from gradcam import GradCAM
from vis import visualize_gradcam
from util import compute_clip_gradcam

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device = "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_NAME = "openai/clip-vit-base-patch32"
print(f"Using device: {device}\nLoading model: {MODEL_NAME}")

In [ ]:
PROMPT = "dog"
IMAGE_PATH = "../data/shiba.png"
OUTPUT_DIR = "../data/"
print(f"Text input: \"{PROMPT}\"\nImage path: {IMAGE_PATH}\nOutput directory: {OUTPUT_DIR}")

In [ ]:
image = Image.open(IMAGE_PATH)
image

In [ ]:
processor = CLIPProcessor.from_pretrained(MODEL_NAME, device_map=device, use_fast=True)
print(f"Loaded processor: {type(processor)}")

In [ ]:
inputs = processor(text=[PROMPT], images=image, return_tensors="pt", padding=True)
print(f"Processed input shapes:\nPixel Values: {inputs['pixel_values'].shape}\nInput Ids: {inputs['input_ids'].shape}\nText Mask: {inputs['attention_mask'].shape}")

In [ ]:
%%time
cams = []
for invert in [False, True]:
    cam, similarity = compute_clip_gradcam(
        model=CLIPModel.from_pretrained(MODEL_NAME),
        inputs=inputs,
        device=device,
        invert_loss=invert,
        debug=False
    )
    cams.append(cam)

In [ ]:
now = datetime.now(ZoneInfo("America/Los_Angeles")).strftime("%Y_%m_%d-%H-%M")
filename = f"{now}-{PROMPT.replace(" ", "-")}.png"
save_path = os.path.join(OUTPUT_DIR, filename)
print(f"Saving to: {save_path}")

In [ ]:
visualize_gradcam(
    original_image=image,
    cams=cams,
    prompt=PROMPT,
    score=similarity,
    save_path=save_path
)